# Importando Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import ast
import pickle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import _pickle as cPickle

# Sistema de Recomendação - Implementação

In [2]:
df=pd.read_csv('movies_dataframe.csv')

## Demographic Filtering

In [3]:
C = df['vote_average'].mean()
C

6.472475494009648

In [4]:
M = df['vote_count'].quantile(0.9)
M

2054.4000000000005

In [5]:
q_movies = df.copy().loc[df['vote_count'] >= M]

In [6]:
q_movies.shape

(643, 31)

In [7]:
def wr(dataframe, m=M, c=C):
  V = dataframe['vote_count']
  R = dataframe['vote_average']
  return ((R*V)+(c*m))/(V+m)

In [8]:
q_movies['score'] =  q_movies.apply(wr, axis=1)

In [9]:
q_movies.head()

,Unnamed: 0,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,...,video,vote_average,vote_count,keywords,cast,crew,return,year,month,score
0,0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,...,False,7.7,5415,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",12.451801,1995.0,10,7.362379
1,1,False,NaN,65000000.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,...,False,6.9,2413,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",4.043035,1995.0,12,6.703397
23,31,False,NaN,29500000.0,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",NaN,63,tt0114746,en,Twelve Monkeys,...,False,7.4,2470,"[{'id': 222, 'name': 'schizophrenia'}, {'id': ...","[{'cast_id': 41, 'character': 'James Cole', 'c...","[{'credit_id': '52fe4212c3a36847f8001ac7', 'de...",5.723390,1995.0,12,6.978838
30,46,False,NaN,33000000.0,"[{'id': 80, 'name': 'Crime'}, {'id': 9648, 'na...",http://www.sevenmovie.com/,807,tt0114369,en,Se7en,...,False,8.1,5915,"[{'id': 476, 'name': 'self-fulfilling prophecy...","[{'cast_id': 17, 'character': 'Detective David...","[{'credit_id': '52fe4279c3a36847f802176f', 'de...",9.918541,1995.0,9,7.680447
32,49,False,NaN,6000000.0,"[{'id': 18, 'name': 'Drama'}, {'id': 80, 'name...",http://www.mgm.com/#/our-titles/2083/The-Usual...,629,tt0114814,en,The Usual Suspects,...,False,8.1,3334,"[{'id': 3703, 'name': 'law'}, {'id': 5493, 'na...","[{'cast_id': 19, 'character': 'Michael McManus...","[{'credit_id': '52fe4260c3a36847f8019a7d', 'de...",3.890261,1995.0,7,7.479484


In [10]:
q_movies[['title', 'vote_count','vote_average', 'score']].head()

,title,vote_count,vote_average,score
0,Toy Story,5415,7.7,7.362379
1,Jumanji,2413,6.9,6.703397
23,Twelve Monkeys,2470,7.4,6.978838
30,Se7en,5915,8.1,7.680447
32,The Usual Suspects,3334,8.1,7.479484


In [11]:
q_movies = q_movies.sort_values('score', ascending = False)

In [12]:
q_movies[['title', 'vote_count','vote_average', 'score']].head(10)

,title,vote_count,vote_average,score
5736,Avengers: Infinity War,24229,8.3,8.157154
5914,Avengers: Endgame,20663,8.3,8.134732
130,The Shawshank Redemption,8358,8.5,8.099963
5854,Spider-Man: Into the Spider-Verse,11043,8.4,8.097657
3119,The Dark Knight,12269,8.3,8.037879
304,The Godfather,6024,8.5,7.984385
1159,Fight Club,9678,8.3,7.979992
120,Pulp Fiction,8670,8.3,7.949914
6420,Spider-Man: No Way Home,10685,8.2,7.921413
6235,Spider-Man: No Way Home,10685,8.2,7.921413


## Usando Similaridade

In [13]:
df.head()

,Unnamed: 0,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,...,title,video,vote_average,vote_count,keywords,cast,crew,return,year,month
0,0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,...,Toy Story,False,7.7,5415,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",12.451801,1995.0,10
1,1,False,NaN,65000000.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,...,Jumanji,False,6.9,2413,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",4.043035,1995.0,12
2,4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",NaN,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,...,Father of the Bride Part II,False,5.7,173,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...","[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",NaN,1995.0,2
3,5,False,NaN,60000000.0,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",NaN,949,tt0113277,en,Heat,...,Heat,False,7.7,1886,"[{'id': 642, 'name': 'robbery'}, {'id': 703, '...","[{'cast_id': 25, 'character': 'Lt. Vincent Han...","[{'credit_id': '52fe4292c3a36847f802916d', 'de...",3.123947,1995.0,12
4,6,False,NaN,58000000.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...",NaN,11860,tt0114319,en,Sabrina,...,Sabrina,False,6.2,141,"[{'id': 90, 'name': 'paris'}, {'id': 380, 'nam...","[{'cast_id': 1, 'character': 'Linus Larrabee',...","[{'credit_id': '52fe44959251416c75039da9', 'de...",0.000000,1995.0,12


In [14]:
df.shape

(6427, 31)

In [15]:
df_movies = df[['id', 'title', 'genres', 'overview', 'keywords', 'cast', 'crew']]

In [16]:
df_movies.shape

(6427, 7)

In [17]:
df_movies.head()

,id,title,genres,overview,keywords,cast,crew
0,862,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","Led by Woody, Andy's toys live happily in his ...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,8844,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",When siblings Judy and Peter discover an encha...,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,11862,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",Just when George Banks has recovered from his ...,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...","[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
3,949,Heat,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...","Obsessive master thief, Neil McCauley leads a ...","[{'id': 642, 'name': 'robbery'}, {'id': 703, '...","[{'cast_id': 25, 'character': 'Lt. Vincent Han...","[{'credit_id': '52fe4292c3a36847f802916d', 'de..."
4,11860,Sabrina,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...",An ugly duckling having undergone a remarkable...,"[{'id': 90, 'name': 'paris'}, {'id': 380, 'nam...","[{'cast_id': 1, 'character': 'Linus Larrabee',...","[{'credit_id': '52fe44959251416c75039da9', 'de..."


In [18]:
df_movies.isnull().sum()

id           0
title        0
genres       0
overview    11
keywords     0
cast         0
crew         0
dtype: int64

In [19]:
df_movies.dropna(inplace=True)

/usr/local/lib/python3.7/dist-packages/pandas/util/_decorators.py:311: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return func(*args, **kwargs)


In [20]:
df_movies.isnull().sum()

id          0
title       0
genres      0
overview    0
keywords    0
cast        0
crew        0
dtype: int64

In [21]:
df_movies.shape

(6416, 7)

In [22]:
def transform(t):
  lista_t = []
  for i in ast.literal_eval(t):
    lista_t.append(i['name'])
  return lista_t

In [23]:
df_movies.head(1)

,id,title,genres,overview,keywords,cast,crew
0,862,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","Led by Woody, Andy's toys live happily in his ...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."


In [24]:
df_movies['genres'] = df_movies['genres'].apply(transform)
df_movies['keywords'] = df_movies['keywords'].apply(transform)
df_movies['cast'] = df_movies['cast'].apply(transform)

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documen

In [25]:
df_movies.head()

,id,title,genres,overview,keywords,cast,crew
0,862,Toy Story,"[Animation, Comedy, Family]","Led by Woody, Andy's toys live happily in his ...","[jealousy, toy, boy, friendship, friends, riva...","[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,8844,Jumanji,"[Adventure, Fantasy, Family]",When siblings Judy and Peter discover an encha...,"[board game, disappearance, based on children'...","[Robin Williams, Jonathan Hyde, Kirsten Dunst,...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,11862,Father of the Bride Part II,[Comedy],Just when George Banks has recovered from his ...,"[baby, midlife crisis, confidence, aging, daug...","[Steve Martin, Diane Keaton, Martin Short, Kim...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
3,949,Heat,"[Action, Crime, Drama, Thriller]","Obsessive master thief, Neil McCauley leads a ...","[robbery, detective, bank, obsession, chase, s...","[Al Pacino, Robert De Niro, Val Kilmer, Jon Vo...","[{'credit_id': '52fe4292c3a36847f802916d', 'de..."
4,11860,Sabrina,"[Comedy, Romance]",An ugly duckling having undergone a remarkable...,"[paris, brother brother relationship, chauffeu...","[Harrison Ford, Julia Ormond, Greg Kinnear, An...","[{'credit_id': '52fe44959251416c75039da9', 'de..."


In [26]:
def find_director(t):
  lista_t = []
  for i in ast.literal_eval(t):
    if i['job'] == 'Director':
      lista_t.append(i['name'])
  return lista_t

In [27]:
df_movies['crew'] = df_movies['crew'].apply(find_director)

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


In [28]:
df_movies.head()

,id,title,genres,overview,keywords,cast,crew
0,862,Toy Story,"[Animation, Comedy, Family]","Led by Woody, Andy's toys live happily in his ...","[jealousy, toy, boy, friendship, friends, riva...","[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",[John Lasseter]
1,8844,Jumanji,"[Adventure, Fantasy, Family]",When siblings Judy and Peter discover an encha...,"[board game, disappearance, based on children'...","[Robin Williams, Jonathan Hyde, Kirsten Dunst,...",[Joe Johnston]
2,11862,Father of the Bride Part II,[Comedy],Just when George Banks has recovered from his ...,"[baby, midlife crisis, confidence, aging, daug...","[Steve Martin, Diane Keaton, Martin Short, Kim...",[Charles Shyer]
3,949,Heat,"[Action, Crime, Drama, Thriller]","Obsessive master thief, Neil McCauley leads a ...","[robbery, detective, bank, obsession, chase, s...","[Al Pacino, Robert De Niro, Val Kilmer, Jon Vo...",[Michael Mann]
4,11860,Sabrina,"[Comedy, Romance]",An ugly duckling having undergone a remarkable...,"[paris, brother brother relationship, chauffeu...","[Harrison Ford, Julia Ormond, Greg Kinnear, An...",[Sydney Pollack]


In [29]:
def replace_empty(l):
  lista_replace_t = []
  for i in l:
    lista_replace_t.append(i.replace(' ',''))
  return lista_replace_t

In [30]:
df_movies['cast'] = df_movies['cast'].apply(replace_empty)
df_movies['crew'] = df_movies['crew'].apply(replace_empty)
df_movies['genres'] = df_movies['genres'].apply(replace_empty)
df_movies['keywords'] = df_movies['keywords'].apply(replace_empty)

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documen

In [31]:
df_movies.head()

,id,title,genres,overview,keywords,cast,crew
0,862,Toy Story,"[Animation, Comedy, Family]","Led by Woody, Andy's toys live happily in his ...","[jealousy, toy, boy, friendship, friends, riva...","[TomHanks, TimAllen, DonRickles, JimVarney, Wa...",[JohnLasseter]
1,8844,Jumanji,"[Adventure, Fantasy, Family]",When siblings Judy and Peter discover an encha...,"[boardgame, disappearance, basedonchildren'sbo...","[RobinWilliams, JonathanHyde, KirstenDunst, Br...",[JoeJohnston]
2,11862,Father of the Bride Part II,[Comedy],Just when George Banks has recovered from his ...,"[baby, midlifecrisis, confidence, aging, daugh...","[SteveMartin, DianeKeaton, MartinShort, Kimber...",[CharlesShyer]
3,949,Heat,"[Action, Crime, Drama, Thriller]","Obsessive master thief, Neil McCauley leads a ...","[robbery, detective, bank, obsession, chase, s...","[AlPacino, RobertDeNiro, ValKilmer, JonVoight,...",[MichaelMann]
4,11860,Sabrina,"[Comedy, Romance]",An ugly duckling having undergone a remarkable...,"[paris, brotherbrotherrelationship, chauffeur,...","[HarrisonFord, JuliaOrmond, GregKinnear, Angie...",[SydneyPollack]


In [32]:
df_movies['overview'] = df_movies['overview'].apply(lambda k: k.split())

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


In [33]:
df_movies.head()

,id,title,genres,overview,keywords,cast,crew
0,862,Toy Story,"[Animation, Comedy, Family]","[Led, by, Woody,, Andy's, toys, live, happily,...","[jealousy, toy, boy, friendship, friends, riva...","[TomHanks, TimAllen, DonRickles, JimVarney, Wa...",[JohnLasseter]
1,8844,Jumanji,"[Adventure, Fantasy, Family]","[When, siblings, Judy, and, Peter, discover, a...","[boardgame, disappearance, basedonchildren'sbo...","[RobinWilliams, JonathanHyde, KirstenDunst, Br...",[JoeJohnston]
2,11862,Father of the Bride Part II,[Comedy],"[Just, when, George, Banks, has, recovered, fr...","[baby, midlifecrisis, confidence, aging, daugh...","[SteveMartin, DianeKeaton, MartinShort, Kimber...",[CharlesShyer]
3,949,Heat,"[Action, Crime, Drama, Thriller]","[Obsessive, master, thief,, Neil, McCauley, le...","[robbery, detective, bank, obsession, chase, s...","[AlPacino, RobertDeNiro, ValKilmer, JonVoight,...",[MichaelMann]
4,11860,Sabrina,"[Comedy, Romance]","[An, ugly, duckling, having, undergone, a, rem...","[paris, brotherbrotherrelationship, chauffeur,...","[HarrisonFord, JuliaOrmond, GregKinnear, Angie...",[SydneyPollack]


In [34]:
df_movies['tags'] = df_movies['genres'] + df_movies['overview'] +  df_movies['keywords'] +  df_movies['cast'] +  df_movies['crew']

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


In [35]:
df_movies.head()

,id,title,genres,overview,keywords,cast,crew,tags
0,862,Toy Story,"[Animation, Comedy, Family]","[Led, by, Woody,, Andy's, toys, live, happily,...","[jealousy, toy, boy, friendship, friends, riva...","[TomHanks, TimAllen, DonRickles, JimVarney, Wa...",[JohnLasseter],"[Animation, Comedy, Family, Led, by, Woody,, A..."
1,8844,Jumanji,"[Adventure, Fantasy, Family]","[When, siblings, Judy, and, Peter, discover, a...","[boardgame, disappearance, basedonchildren'sbo...","[RobinWilliams, JonathanHyde, KirstenDunst, Br...",[JoeJohnston],"[Adventure, Fantasy, Family, When, siblings, J..."
2,11862,Father of the Bride Part II,[Comedy],"[Just, when, George, Banks, has, recovered, fr...","[baby, midlifecrisis, confidence, aging, daugh...","[SteveMartin, DianeKeaton, MartinShort, Kimber...",[CharlesShyer],"[Comedy, Just, when, George, Banks, has, recov..."
3,949,Heat,"[Action, Crime, Drama, Thriller]","[Obsessive, master, thief,, Neil, McCauley, le...","[robbery, detective, bank, obsession, chase, s...","[AlPacino, RobertDeNiro, ValKilmer, JonVoight,...",[MichaelMann],"[Action, Crime, Drama, Thriller, Obsessive, ma..."
4,11860,Sabrina,"[Comedy, Romance]","[An, ugly, duckling, having, undergone, a, rem...","[paris, brotherbrotherrelationship, chauffeur,...","[HarrisonFord, JuliaOrmond, GregKinnear, Angie...",[SydneyPollack],"[Comedy, Romance, An, ugly, duckling, having, ..."


In [36]:
df3 = df_movies.drop(columns = ['genres', 'overview', 'keywords', 'cast', 'crew'])

In [37]:
df3.head()

,id,title,tags
0,862,Toy Story,"[Animation, Comedy, Family, Led, by, Woody,, A..."
1,8844,Jumanji,"[Adventure, Fantasy, Family, When, siblings, J..."
2,11862,Father of the Bride Part II,"[Comedy, Just, when, George, Banks, has, recov..."
3,949,Heat,"[Action, Crime, Drama, Thriller, Obsessive, ma..."
4,11860,Sabrina,"[Comedy, Romance, An, ugly, duckling, having, ..."


In [38]:
df3['tags'] = df3['tags'].apply(lambda i: ' '.join(i))

In [39]:
df3.head()

,id,title,tags
0,862,Toy Story,"Animation Comedy Family Led by Woody, Andy's t..."
1,8844,Jumanji,Adventure Fantasy Family When siblings Judy an...
2,11862,Father of the Bride Part II,Comedy Just when George Banks has recovered fr...
3,949,Heat,Action Crime Drama Thriller Obsessive master t...
4,11860,Sabrina,Comedy Romance An ugly duckling having undergo...


In [40]:
df3 = df3.reset_index(drop=True)

In [41]:
df3.head()

,id,title,tags
0,862,Toy Story,"Animation Comedy Family Led by Woody, Andy's t..."
1,8844,Jumanji,Adventure Fantasy Family When siblings Judy an...
2,11862,Father of the Bride Part II,Comedy Just when George Banks has recovered fr...
3,949,Heat,Action Crime Drama Thriller Obsessive master t...
4,11860,Sabrina,Comedy Romance An ugly duckling having undergo...


In [42]:
vectorizer = CountVectorizer(max_features=5000, stop_words='english')

In [43]:
vectorizer

CountVectorizer(max_features=5000, stop_words='english')

In [44]:
X = vectorizer.fit_transform(df3['tags']).toarray()

In [45]:
X.shape

(6416, 5000)

In [46]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [47]:
similarity = cosine_similarity(X)

In [48]:
similarity

array([[1.        , 0.03928371, 0.02125256, ..., 0.05360563, 0.        ,
        0.        ],
       [0.03928371, 1.        , 0.        , ..., 0.01895245, 0.04811252,
        0.0243975 ],
       [0.02125256, 0.        , 1.        , ..., 0.02050662, 0.        ,
        0.        ],
       ...,
       [0.05360563, 0.01895245, 0.02050662, ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.04811252, 0.        , ..., 0.        , 1.        ,
        0.05634362],
       [0.        , 0.0243975 , 0.        , ..., 0.        , 0.05634362,
        1.        ]])

In [49]:
def sistema_recomendacao(movie):
    index = df3.loc[df3['title'] == movie].index[0]
    distance = sorted(list(enumerate(similarity[index])), reverse=True, key = lambda x:x[1])
    for i in distance[1:6]:
        print(df3.iloc[i[0]].title)

In [50]:
pickle.dump(df3, open('lista_filmes.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))